# GraphSAGE 학습 — Colab
XGBoost 베이스라인과 같은 분할·지표·시드(0~4)로 GraphSAGE를 학습하고 비교합니다.


In [ ]:
# 1) 환경 확인
!python --version
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 T4 GPU로 바꾸세요'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) 저장소 가져오기 (private이면 GitHub 토큰 입력, public이면 엔터)
import getpass, os
if not os.path.exists('2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git
%cd 2026-2-HAI-ProjectTeam8
!git fetch -q origin && git checkout -q sage && git pull -q origin sage || true  # PR 머지 전에는 sage 브랜치 사용
!ls models common

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/elliptic
!cp "/content/drive/MyDrive/Colab Notebooks/elliptic/"*.csv data/elliptic/
!ls -la data/elliptic

In [ ]:
# 4) PyTorch Geometric 설치 (Colab에는 torch가 이미 있음)
!pip install -q torch-geometric
import torch_geometric; print('torch_geometric', torch_geometric.__version__)

In [ ]:
# 5) 데이터 검증 (모두 OK여야 함)
!python scripts/verify_data.py data/elliptic

In [ ]:
# 6) 빠른 점검: 시드 1개, 5 에폭 (몇십 초). 에러 없이 '저장:' 까지 나오면 통과
!python -m models.sage --data-root data/elliptic --seeds 0 --max-epochs 5 --results-dir results_smoke

In [ ]:
# 7) 본 학습: 시드 0~4 (조기 종료 patience 30). 결과는 results/ 에 저장
!python -m models.sage --data-root data/elliptic

In [ ]:
# 8) XGBoost와 비교표 (repo의 results/metrics/xgb.csv 가 있으면 함께 표시)
!python scripts/collect_results.py

In [ ]:
# 9) 시간 단계별 F1: GCN vs XGBoost (35~42 vs 43~49 구간 비교)
import pandas as pd, os
pd.set_option('display.width', 200)
fs = {m: f'results/timestep_f1_{m}.csv' for m in ('xgb', 'gcn', 'sage')}
t = {m: pd.read_csv(p).groupby('time_step').f1.mean().round(3) for m, p in fs.items() if os.path.exists(p)}
df = pd.DataFrame(t)
print(df.to_string())
print('\n평균 F1  35~42:', df.loc[35:42].mean().round(3).to_dict(), '| 43~49:', df.loc[43:49].mean().round(3).to_dict())

In [ ]:
# 10) 결과를 드라이브에 백업 (런타임이 끊겨도 보존). emb는 용량이 커서 git에는 안 올라갑니다
!mkdir -p "/content/drive/MyDrive/Colab Notebooks/elliptic/sage_results"
!cp -r results/metrics results/pred results/emb results/timestep_f1_sage.csv results/summary.csv "/content/drive/MyDrive/Colab Notebooks/elliptic/sage_results/"
!ls "/content/drive/MyDrive/Colab Notebooks/elliptic/sage_results"